<a href="https://colab.research.google.com/github/galipaydin/pes517/blob/main/demo/demo_pose_live.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## What this produces

| File | What you show in class |
|---|---|
| `demo_side_by_side.png` | One frame: raw on the left, skeleton on the right |
| `demo_overlay.mp4` | The clip with the skeleton drawn on the athlete |
| `demo_knee_angle.png` | The knee-angle curve — three squats, three dips |

> `run_demo.py` in the same folder names its outputs after the clip instead
> (`squat_frontal_raise_knee_angle.png` and so on), so running it on a second clip
> never overwrites the first. This notebook writes one set, so re-running it on a
> different clip **does** replace them — rename or move them if you want to keep both.


---
## 1 — Install

Deliberately **not pinned**. MediaPipe is at 1.x and the Tasks API is stable across it; pinning to an old
0.10.x would pin you to the dying interface. If a future release does break this, the fix is in section 9.


In [ ]:
!pip install -q mediapipe opencv-python-headless

import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
from pathlib import Path

print('mediapipe', mp.__version__, '| opencv', cv2.__version__)
assert hasattr(mp, 'tasks'), 'This build has no Tasks API — see section 9.'
print('Tasks API available ✓')


---
## 2 — Download the model

The Tasks API keeps the model in a separate `.task` file that you fetch once. Three sizes exist;
**full** is the right trade-off for a squat on a CPU.

| Model | Size | When to use |
|---|---|---|
| `lite` | smallest | Live webcam on a weak machine |
| **`full`** | middle | **This demo, and Week 10** |
| `heavy` | largest | When accuracy matters more than time |


In [ ]:
MODEL = 'pose_landmarker_full.task'
URL = ('https://storage.googleapis.com/mediapipe-models/pose_landmarker/'
       'pose_landmarker_full/float16/latest/pose_landmarker_full.task')

if not Path(MODEL).exists():
    !wget -q -O {MODEL} {URL}

size_mb = Path(MODEL).stat().st_size / 1e6
print(f'{MODEL}: {size_mb:.1f} MB')
assert size_mb > 1, 'Download failed — check the network, or see section 9.'


---
## 3 — Get a clip

**The quickest path: use one of the four clips that ship with this course.** They download in a second
and they are already known to track well, so you can see the whole thing work before you involve your
own footage.

| Clip | What it is | How it tracked |
|---|---|---|
| **`squat_frontal_raise`** | Kettlebell goblet squats, side on, barefoot, whole body | **0.94 confidence · 3 reps · 134° range** |
| `squat_single_leg` | Single-leg squat outdoors, very clean background | 0.99 confidence · 1 rep |
| `squat_demo` | Barbell back squat — **feet cut off at the bottom of frame** | 0.78 · a worked example of bad filming |
| `sports2d_demo` | Walking in the sagittal plane, backlit, several people | 0.92 · deliberately difficult |

Licences and attribution for the clips are in the README next to this notebook. `squat_frontal_raise` is the one to start with.


In [ ]:
CLIP = 'squat_frontal_raise'   # ✏️ EDIT ME — any name from the table above

import urllib.request
from pathlib import Path

SRC = f'{CLIP}.mp4'
if not Path(SRC).exists():
    url = f'https://raw.githubusercontent.com/galipaydin/pes517/main/demo/clips/{SRC}'
    urllib.request.urlretrieve(url, SRC)
print(f'{SRC} — {Path(SRC).stat().st_size/1e6:.1f} MB')


### 3b — Or upload your own

Better, if you have it: your own video.

Film it side on, with the whole body including the feet in frame, the camera on something fixed,
15 seconds at most and 720p at most.

Run this cell instead of the one above.


In [ ]:
from google.colab import files
up = files.upload()
SRC = list(up.keys())[0]
print('using', SRC)


### 3c — Last resort: a synthetic clip

Only if you have no network and no footage. It writes a drawn figure so you can confirm the
notebook runs. **Pose estimation on a stick figure is unreliable** — use it to test the pipeline,
never to teach.


In [ ]:
if 'SRC' not in globals():
    SRC, FPS_SYN, N = 'placeholder.mp4', 30, 180
    w, h = 720, 540
    vw = cv2.VideoWriter(SRC, cv2.VideoWriter_fourcc(*'mp4v'), FPS_SYN, (w, h))
    for i in range(N):
        f = np.full((h, w, 3), 232, np.uint8)
        d = int(70 * max(0.0, np.sin(np.pi * (i % 60) / 60)))      # three squats
        ankle = (360, 470); knee = (360 - d//3, 380 + d//2); hip = (360, 280 + d)
        shoulder = (360, 190 + d); head = (360, 150 + d)
        cv2.circle(f, head, 28, (70, 70, 70), -1)
        for a, b in [(shoulder, hip), (hip, knee), (knee, ankle)]:
            cv2.line(f, a, b, (70, 70, 70), 14)
        cv2.line(f, shoulder, (shoulder[0]+70, shoulder[1]+60), (70,70,70), 10)
        vw.write(f)
    vw.release()
    print('wrote placeholder.mp4 — replace with real footage before the session')


### 3d — Trim and downscale (run this if your clip is long or high-resolution)

Stock footage is usually **4K and 20–30 seconds**. On a free Colab CPU that is several minutes of
processing for no benefit — the pose model works on a much smaller image internally.

This cell cuts the clip to the first N seconds and scales it down. A phone clip you filmed at 720p can
skip it.

**Vertical video is fine for the model** — arguably better, since the body fills the frame. It is only
awkward on a 16:9 projector, and the side-by-side still in section 6 solves that.


In [ ]:
MAX_SECONDS = 10     # ✏️ how much of the clip to keep
TARGET_H    = 720    # ✏️ height in pixels; width scales automatically

import subprocess
probe = cv2.VideoCapture(SRC)
w0 = int(probe.get(cv2.CAP_PROP_FRAME_WIDTH))
h0 = int(probe.get(cv2.CAP_PROP_FRAME_HEIGHT))
n0 = int(probe.get(cv2.CAP_PROP_FRAME_COUNT))
f0 = probe.get(cv2.CAP_PROP_FPS) or 30
probe.release()
print(f'original: {w0}x{h0}, {n0/f0:.1f}s, {f0:.0f} fps')

if h0 > TARGET_H or n0/f0 > MAX_SECONDS:
    TRIMMED = 'clip_small.mp4'
    subprocess.run(['ffmpeg','-loglevel','error','-y','-i',SRC,
                    '-t',str(MAX_SECONDS),
                    '-vf',f'scale=-2:{TARGET_H}',
                    '-an', TRIMMED], check=True)
    SRC = TRIMMED
    chk = cv2.VideoCapture(SRC)
    print(f'trimmed : {int(chk.get(3))}x{int(chk.get(4))}, '
          f'{chk.get(7)/(chk.get(5) or 30):.1f}s  ->  using {SRC}')
    chk.release()
else:
    print('already small enough — no change')


---
## 4 — Run the pose landmarker

This is the whole thing. Note `RunningMode.VIDEO` — it tells the model the frames are a sequence, so it
tracks between them instead of searching the whole image every time. That is both faster and steadier.


In [ ]:
BaseOptions   = mp_python.BaseOptions
PoseLandmarker        = vision.PoseLandmarker
PoseLandmarkerOptions = vision.PoseLandmarkerOptions
RunningMode           = vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL),
    running_mode=RunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_tracking_confidence=0.5)

# MediaPipe landmark indices (33 total)
R_HIP, R_KNEE, R_ANKLE, R_HEEL, R_FOOT = 24, 26, 28, 30, 32
EDGES = [(11,12),(11,23),(12,24),(23,24),          # torso
         (11,13),(13,15),(12,14),(14,16),          # arms
         (23,25),(25,27),(27,29),(29,31),(27,31),  # left leg + foot
         (24,26),(26,28),(28,30),(30,32),(28,32)]  # right leg + foot

def draw(img, lms):
    h, w = img.shape[:2]
    pt = lambda i: (int(lms[i].x * w), int(lms[i].y * h))
    for a, b in EDGES:
        cv2.line(img, pt(a), pt(b), (255, 255, 255), 5, cv2.LINE_AA)
        cv2.line(img, pt(a), pt(b), (43, 100, 27), 3, cv2.LINE_AA)   # teal, BGR
    for i in range(11, 33):
        cv2.circle(img, pt(i), 6, (255, 255, 255), -1, cv2.LINE_AA)
        cv2.circle(img, pt(i), 4, (43, 28, 138), -1, cv2.LINE_AA)    # maroon, BGR
    return img

cap = cv2.VideoCapture(SRC)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter('demo_overlay.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (W, H))

rows, still_raw, still_ovl = [], None, None
with PoseLandmarker.create_from_options(options) as landmarker:
    idx = 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB,
                          data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        res = landmarker.detect_for_video(mp_img, int(idx / fps * 1000))
        drawn = frame.copy()
        if res.pose_landmarks:
            lms = res.pose_landmarks[0]
            drawn = draw(drawn, lms)
            rows.append(dict(t=idx/fps,
                hip_x=lms[R_HIP].x,   hip_y=lms[R_HIP].y,
                knee_x=lms[R_KNEE].x, knee_y=lms[R_KNEE].y,
                ankle_x=lms[R_ANKLE].x, ankle_y=lms[R_ANKLE].y,
                vis=min(lms[R_HIP].visibility, lms[R_KNEE].visibility, lms[R_ANKLE].visibility)))
            if still_raw is None and idx > fps:
                still_raw, still_ovl = frame.copy(), drawn.copy()
        out.write(drawn)
        idx += 1
cap.release(); out.release()

pct = 100 * len(rows) / max(idx, 1)
print(f'{idx} frames at {fps:.0f} fps · landmarks found in {len(rows)} ({pct:.0f}%)')
if pct < 70:
    print('\n⚠️  Patchy detection. Usually the whole body is not in frame, the background is busy,')
    print('    or the person is too small in the picture.')


---
## 5 — The knee angle

Three points make an angle. An angle over time makes a movement.


In [ ]:
df = pd.DataFrame(rows)

thigh = np.c_[df.hip_x   - df.knee_x, df.hip_y   - df.knee_y]
shank = np.c_[df.ankle_x - df.knee_x, df.ankle_y - df.knee_y]
cos = (thigh*shank).sum(1) / (np.linalg.norm(thigh,axis=1) * np.linalg.norm(shank,axis=1))
df['knee_flex_deg'] = 180 - np.degrees(np.arccos(np.clip(cos, -1, 1)))
df['smooth'] = df.knee_flex_deg.rolling(5, center=True, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.plot(df.t, df.smooth, color='#8A1C2B', lw=2.4)
ax.fill_between(df.t, df.smooth.min(), df.smooth, color='#8A1C2B', alpha=.08)
ax.set_xlabel('Time (seconds)', fontsize=12)
ax.set_ylabel('Knee flexion (degrees)', fontsize=12)
ax.set_title('Knee angle, measured from an ordinary phone video', fontsize=14, pad=12)
ax.grid(alpha=.3)
for s in ('top','right'): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig('demo_knee_angle.png', dpi=160); plt.show()

print(f'Peak knee flexion : {df.smooth.max():.0f}°')
print(f'Range of motion   : {df.smooth.max() - df.smooth.min():.0f}°')
print(f'Frames measured   : {len(df)}')
print(f'Lowest confidence : {df.vis.min():.2f}   (below ~0.5 means the model was guessing)')


---
## 6 — One frame, before and after

This still is what you put on screen while you talk. It makes the point faster than the video does.


In [ ]:
if still_raw is not None:
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    ax[0].imshow(cv2.cvtColor(still_raw, cv2.COLOR_BGR2RGB))
    ax[0].set_title('What the camera recorded', fontsize=13)
    ax[1].imshow(cv2.cvtColor(still_ovl, cv2.COLOR_BGR2RGB))
    ax[1].set_title('What the computer found', fontsize=13)
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.savefig('demo_side_by_side.png', dpi=160); plt.show()
else:
    print('No frame captured — detection failed. See section 9.')


---
## 7 — Check it plays, then download

Colab will not play the raw mp4, so this re-encodes it. **In class, play the downloaded file in your own
video player** — one less thing to go wrong.


In [ ]:
from IPython.display import HTML
from base64 import b64encode

!ffmpeg -loglevel error -y -i demo_overlay.mp4 -vcodec libx264 -pix_fmt yuv420p demo_overlay_web.mp4
data = b64encode(open('demo_overlay_web.mp4','rb').read()).decode()
HTML(f'<video width=640 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')


In [ ]:
from google.colab import files
for f in ['demo_overlay_web.mp4', 'demo_knee_angle.png', 'demo_side_by_side.png']:
    try: files.download(f)
    except Exception as e: print('could not download', f, '—', e)


---
## 8 — Optional: the two-line version, and a licence lesson

**YOLO26-pose** (Ultralytics, January 2026) does pose estimation in two lines and handles several people
at once. It reports roughly +7 AP over YOLO11 on the COCO pose benchmark.

Run this only if you want a second demo. It is a larger install and it downloads weights on first use.

### Why it is *not* the primary demo here

| | |
|---|---|
| **17 keypoints, not 33** | COCO convention stops at the ankle. No heel, no foot index — worse for a squat. |
| **AGPL-3.0** | Build a tool on it and distribute it, and you must open-source your work, or buy a commercial licence. |

That second row is worth three minutes of class time. It is a concrete, current instance of exactly what
Week 13 and Week 14 are about: **the thing that stops a club deploying a model is frequently not accuracy,
it is the licence and the contract.** Students who have met this in Week 1 recognise it later.


In [ ]:
# !pip install -q ultralytics
# from ultralytics import YOLO
#
# model = YOLO('yolo26n-pose.pt')        # downloads on first use
# results = model(SRC, save=True, conf=0.5)
#
# # COCO-17 indices: 11/12 hips · 13/14 knees · 15/16 ankles
# k = results[0].keypoints.xy[0]
# print('keypoints in frame 1:', k.shape)
# print('annotated video saved under runs/pose/predict/')


---
## 9 — Troubleshooting

| Problem | What to do |
|---|---|
| `AttributeError: module 'mediapipe' has no attribute 'tasks'` | Very old build. `!pip install -q --upgrade mediapipe`, then restart the runtime. |
| It aborts with `DrishtiMetalHelper` / `Service is unavailable` | You are not on Colab — this is the MediaPipe 1.0.x fault on macOS Apple silicon. `pip install 'mediapipe==0.10.18'`, which has the same Tasks API. Colab is Linux and unaffected. |
| You find a tutorial using `mp.solutions.pose` | That is the dead legacy API. Ignore it; this notebook uses the current one. |
| Model download fails | Check the network. The three URLs are in section 2; try `lite` if `full` will not come down. |
| Landmarks found in under 70% of frames | Whole body not in frame, busy background, or the athlete is too small. Re-film with the whole body in frame against a plainer background. |
| The skeleton flickers | Low light or low frame rate. Film brighter, or at 60 fps. Raising `min_tracking_confidence` does not help here. |
| `vis` is below 0.5 | The model is guessing at those landmarks. Do not quote an angle from those frames. |
| Knee curve is noisy | Increase the `rolling(5)` window to 9 or 11. |
| Video will not play in class | Use `demo_overlay_web.mp4`, not `demo_overlay.mp4`. |
| Wrong leg tracked | Indices 24/26/28 are the **right** side. Left is 23/25/27. |
| Everything is slow | The clip is too long or too large. 15 seconds, 720p. Or switch to the `lite` model. |

---

## What changed, and when

| | |
|---|---|
| **October 2026** | Rewritten for the MediaPipe **Tasks API**. The previous version used `mp.solutions.pose`, the legacy interface Google stopped supporting in March 2023 and which is being removed. Added the YOLO26-pose aside. |

If you are reading this a year from now: run section 1 and check the printed version. The Tasks API has
been stable since 0.10.0 and through 1.x, so this should still run. If it does not, the pattern to search
for is `mediapipe tasks vision PoseLandmarker python`.
